# Type 2 (object-level concept intervention): evaluation, COCO vocabulary

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch

from utils import obj_enc_aggregation
from coco_category_vocab import load_category_names
from intervention_image_level import (
    load_intervention_testset, load_gt_categories, load_category_id_by_name, load_checkpoint,
    load_variants, load_boundary_type, apply_intervention,
    balanced_accuracy, positive_variant_accuracy, nb_accuracy,
)
from intervention_object_level_existence import load_coco_gt_boxes, load_original_image_sizes
from intervention_object_level_concept_correction_coco import (
    load_objects_raw, match_image_objects, apply_object_level_concept_correction,
)

BASE = "../encodings/cocologic/clip/splice_embeddings_0.25_coco_categories.h5"
OBJ = "../encodings/cocologic/obj_encodings/clip/obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25_coco_categories.h5"
TRAIN_JSON = "../data/cocologic/cocologic_train.json"
TEST_JSON = "../data/cocologic/cocologic_test.json"
TRAIN_INSTANCES = "../datasets/coco/annotations/instances_train2017.json"
TEST_INSTANCES = "../datasets/coco/annotations/instances_val2017.json"
K = 7

category_names = load_category_names(TEST_INSTANCES)  
category_id_by_name = load_category_id_by_name(TEST_INSTANCES)
dim_by_category_id = {category_id_by_name[name]: i for i, name in enumerate(category_names)}
cat_ids = [category_id_by_name[name] for name in category_names]  # column of each dimension in gt vectors
PERSON = dim_by_category_id[category_id_by_name["person"]]

targets0 = torch.load("../data/cocologic/percentile_targets_full_train_sum_coco_categories.pt")
targets2 = torch.load("../data/cocologic/object_level_percentile_targets_coco_categories.pt")

for name, t in targets0.items():
    assert list(t["boost"]) == [dim_by_category_id[category_id_by_name[name]]]
boost0 = torch.tensor([targets0[name]["boost"][d] for d, name in enumerate(category_names)])
suppress0 = torch.tensor([targets0[name]["suppress"][d] for d, name in enumerate(category_names)])
boost2 = torch.tensor([targets2["boost"][d] for d in range(80)])
suppress2 = torch.tensor([targets2["suppress"][d] for d in range(80)])

model = load_checkpoint("../checkpoints/cocologic/sum_super_multilabel_seed42_coco_categories.pt")

In [ ]:
emb, obj, img_ids, labels = load_intervention_testset(BASE, OBJ, K)
labels = labels.float()
variants = load_variants(TEST_JSON, img_ids)
boundary_type = load_boundary_type(TEST_JSON, img_ids)
gt_test = load_gt_categories(TEST_JSON, img_ids)  
present = gt_test[:, cat_ids] > 0  

base = obj_enc_aggregation("sum", obj, emb, only_objects=False)

raw_emb, raw_boxes, raw_ids = load_objects_raw(OBJ, "test")
raw_idx = {iid.item(): i for i, iid in enumerate(raw_ids)}
gt_boxes = load_coco_gt_boxes(TEST_INSTANCES)  # crowd excluded
orig_sizes = load_original_image_sizes(TEST_INSTANCES)

matches = []
for iid in img_ids.flatten().tolist():
    boxes = raw_boxes[raw_idx[iid]]
    gts = gt_boxes.get(iid, [])
    pairs, _ = match_image_objects(boxes, gts, orig_sizes[iid], 0.5, "greedy") if gts else ([], None)
    matches.append(pairs)

reached = torch.zeros_like(present)
for i, pairs in enumerate(matches):
    for _, cat in pairs:
        reached[i, dim_by_category_id[cat]] = True
assert not (reached & ~present).any()

def predict(x):
    with torch.no_grad():
        return model(x)

def pad(t):
    out = torch.zeros(K, t.shape[1])
    out[:t.shape[0]] = t
    return out

def rule_mean(logits):
    return {
        "B-Acc": 100 * torch.nanmean(balanced_accuracy(logits, labels)).item(),
        "NB-Acc": 100 * torch.nanmean(nb_accuracy(logits, boundary_type)).item(),
        "PosVar-Acc": 100 * torch.nanmean(positive_variant_accuracy(logits, variants)).item(),
    }

In [ ]:
# Type 2 with the existing function
type2_obj, n_matched = [], []
for iid in img_ids.flatten().tolist():
    corrected, n = apply_object_level_concept_correction(
        raw_emb[raw_idx[iid]], raw_boxes[raw_idx[iid]], gt_boxes.get(iid, []), orig_sizes[iid],
        targets2, dim_by_category_id,
    )
    type2_obj.append(pad(corrected))
    n_matched.append(n)
type2 = obj_enc_aggregation("sum", torch.stack(type2_obj), emb, only_objects=False)
n_matched = torch.tensor(n_matched)


## Accuracy



In [ ]:
# typ 2 comparison function for Type 2 with a different boost vector
def type2_variant(own_boost):
    out = []
    for i, iid in enumerate(img_ids.flatten().tolist()):
        corrected = raw_emb[raw_idx[iid]].clone()
        for p, cat in matches[i]:
            d = dim_by_category_id[cat]
            corrected[p] = suppress2
            corrected[p, d] = own_boost[d]
        out.append(pad(corrected))
    return obj_enc_aggregation("sum", torch.stack(out), emb, only_objects=False)

def type0(boost_mask):
    out = base.clone()
    out = torch.where(~present, suppress0, out)
    out = torch.where(boost_mask, boost0, out)
    return out

assert torch.allclose(type2_variant(boost2), type2)

type2_t0values = type2_variant(boost0)
type0_restricted = type0(reached)
type0_full = type0(present)
assert torch.equal(type0_full, apply_intervention(base, gt_test, targets0, category_id_by_name, suppress=True))

conditions = {
    "a) Baseline": base,
    "b) Type 2": type2,
    "c) Type 2 with Type 0 values": type2_t0values,
    "d) Type 0 restricted to Type 2 coverage": type0_restricted,
    "e) Type 0 (full)": type0_full,
}
table1 = pd.DataFrame({name: rule_mean(predict(x)) for name, x in conditions.items()}).T
print(table1.round(2).to_string())

## Coverage


In [ ]:
n_reached, n_present = reached.sum().item(), present.sum().item()
print(f"(image, category) pairs with GT presence reached by Type 2: {n_reached} / {n_present} = {100 * n_reached / n_present:.1f} %")

## Intervention values


In [ ]:
train_emb, train_obj, train_ids, _ = load_intervention_testset(BASE, OBJ, K, split="train")
gt_train = load_gt_categories(TRAIN_JSON, train_ids)
present_train = gt_train[:, cat_ids] > 0

train_agg = obj_enc_aggregation("sum", train_obj, train_emb, only_objects=False)
p95_type0 = torch.tensor([np.percentile(train_agg[present_train[:, d], d].numpy(), 95) for d in range(80)])
assert torch.allclose(p95_type0, boost0)

p95_image = torch.tensor([np.percentile(train_emb[present_train[:, d], d].numpy(), 95) for d in range(80)])

tr_emb, tr_boxes, tr_ids = load_objects_raw(OBJ, "train")
gt_boxes_train = load_coco_gt_boxes(TRAIN_INSTANCES)
sizes_train = load_original_image_sizes(TRAIN_INSTANCES)

matched_vecs, matched_dims = [], []
for e, b, iid in zip(tr_emb, tr_boxes, tr_ids.flatten().tolist()):
    gts = gt_boxes_train.get(iid, [])
    if not gts:
        continue
    pairs, _ = match_image_objects(b, gts, sizes_train[iid], 0.5, "greedy")
    for p, cat in pairs:
        d = dim_by_category_id[cat]
        matched_vecs.append(e[p])
        matched_dims.append(d)
matched_vecs = torch.stack(matched_vecs).numpy()
matched_dims = np.array(matched_dims)

p95_type2 = torch.tensor([np.percentile(matched_vecs[matched_dims == d, d], 95) for d in range(80)])
p5_type2 = torch.tensor([np.percentile(matched_vecs[matched_dims != d, d], 5) for d in range(80)])
assert torch.allclose(p95_type2, boost2) and torch.allclose(p5_type2, suppress2)

print(f"image-level boost (P95):  {np.median(p95_type0.numpy()):.3f}")
print(f"object-level boost (P95): {np.median(p95_type2.numpy()):.3f}")
print(f"full-image P95:           {np.median(p95_image.numpy()):.3f}")
